# Day 4 — Statistics for Machine Learning
**30-Day ML & AI Mastery Plan · Shubham**

Day 3 asked "if I know how the world works, how likely is this data?" Statistics asks the reverse: **"I have this data. What can I honestly say about the world?"** Every time you report a model's accuracy, compare two models, scale a feature or run an A/B test, you are doing statistics.

| Part | What you'll learn | Time |
|---|---|---|
| 1 | Centre: mean, median, mode, trimmed mean | 15 min |
| 2 | Spread: range, variance, standard deviation, IQR | 15 min |
| 3 | Shape: skewness and the log transform | 10 min |
| **4** | **Live demo 1: normalisation vs standardisation on real features** | 30 min |
| 5 | Population vs sample, sampling methods and bias | 15 min |
| **6** | **Live demo 2: the Central Limit Theorem by simulation** | 25 min |
| 7 | Confidence intervals (and what "95%" really means) | 25 min |
| 8 | Hypothesis tests: p-values, errors, power, the right test | 40 min |
| 9 | An A/B test, end to end | 20 min |
| 10 | Exercises | 30–60 min |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(seed=4)

titanic = sns.load_dataset("titanic")
tips = sns.load_dataset("tips")          # 244 restaurant bills: total, tip, smoker, day, time, size
print(titanic.shape, tips.shape)

---
# Part 1 — Centre: where is the "middle" of the data?

| Measure | How | Strength | Weakness |
|---|---|---|---|
| **Mean** | sum ÷ count | uses every value; the basis of most maths | pulled by outliers |
| **Median** | middle value after sorting | robust to outliers | ignores how extreme the extremes are |
| **Mode** | most frequent value | the only one that works for categories | can be several, or meaningless for continuous data |
| **Trimmed mean** | mean after dropping the top and bottom x% | a compromise | you must choose x |

In [ ]:
fare = titanic["fare"]
print("mean   :", round(fare.mean(), 2))
print("median :", round(fare.median(), 2))
print("mode   :", fare.mode()[0])
print("10% trimmed mean:", round(stats.trim_mean(fare, 0.10), 2))
print("embark_town mode:", titanic["embark_town"].mode()[0])   # mode is the only centre for categories

The mean fare (32.2) is more than **double** the median (14.5): a handful of very expensive tickets pull the mean up. When mean ≫ median, the data is **right-skewed** and the median is the honest "typical" value. That's why house prices and salaries are reported as medians.

See it: one billionaire walks into a café.

In [ ]:
incomes = np.array([30, 35, 40, 42, 45, 50, 55])          # thousands per year
print("mean before:", incomes.mean().round(1), "| median before:", np.median(incomes))
with_billionaire = np.append(incomes, 1_000_000)
print("mean after: ", with_billionaire.mean().round(1), "| median after: ", np.median(with_billionaire))

---
# Part 2 — Spread: how far apart are the values?

- **Range** = max − min. Uses only 2 values, so one outlier controls it.
- **Variance** = average squared distance from the mean. Units are squared (₹²), which is hard to read.
- **Standard deviation (σ, s)** = √variance, back in the original units. "Typically, values sit about 1 std from the mean."
- **IQR** = Q3 − Q1, the width of the middle 50%. Robust, like the median.

**Population vs sample variance.** Divide by **n** for a whole population, by **n − 1** for a sample (Bessel's correction). Why? Distances are measured from the *sample* mean, which sits closer to the sample than the true mean does, so dividing by n underestimates the true spread. Let's prove it by simulation.

In [ ]:
x = np.array([2, 4, 4, 4, 5, 5, 7, 9])
print("np.std  (divides by n,   ddof=0):", np.std(x))
print("np.std  (divides by n-1, ddof=1):", round(np.std(x, ddof=1), 3))
print("pandas .std() default is ddof=1:", round(pd.Series(x).std(), 3))

# Simulation: true variance of N(0, 10) is 100. Take 100,000 samples of size 5.
samples = rng.normal(0, 10, size=(100_000, 5))
print("\naverage variance dividing by n  :", samples.var(axis=1, ddof=0).mean().round(1), " <- too small")
print("average variance dividing by n-1:", samples.var(axis=1, ddof=1).mean().round(1), " <- unbiased, close to 100")

In [ ]:
q1, q3 = fare.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}")
print(f"outlier fences: below {low:.2f} or above {high:.2f}")
print("fares flagged as outliers:", ((fare < low) | (fare > high)).sum(), "of", len(fare))

# z-score method for comparison: |z| > 3
z = (fare - fare.mean()) / fare.std()
print("flagged by |z| > 3:", (z.abs() > 3).sum())

The two outlier rules disagree a lot (116 vs 20). On skewed data the z-score rule is weak, because the outliers inflate the std that's supposed to detect them. **Use IQR for skewed data, z-scores for roughly normal data**, and always look at a plot before deleting anything.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
sns.boxplot(data=titanic, x="pclass", y="fare", ax=axes[0])
axes[0].set_title("Fare by class: box = IQR, line = median, dots = beyond 1.5 IQR")
sns.histplot(fare, bins=50, ax=axes[1])
for value, label, colour in [(fare.mean(), "mean", "red"), (fare.median(), "median", "green")]:
    axes[1].axvline(value, color=colour, linestyle="--", label=f"{label} {value:.1f}")
axes[1].legend(); axes[1].set_title("Right-skewed: mean is pulled right of the median")
plt.tight_layout(); plt.show()

---
# Part 3 — Shape: skewness and kurtosis

- **Skewness** = 0 for symmetric data, > 0 for a long right tail (income, fares, waiting times), < 0 for a long left tail (exam scores when most people do well).
- **Kurtosis** measures how heavy the tails are: how often you see extreme values compared with a normal distribution. pandas reports **excess** kurtosis, where normal = 0.

**Fix for right skew:** `np.log1p(x)` = log(1 + x). It squeezes big values much more than small ones.

In [ ]:
log_fare = np.log1p(fare)
print("skewness  fare:", round(fare.skew(), 2), "-> log1p(fare):", round(log_fare.skew(), 2))
print("kurtosis  fare:", round(fare.kurt(), 2), "-> log1p(fare):", round(log_fare.kurt(), 2))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
sns.histplot(fare, bins=40, ax=axes[0]); axes[0].set_title(f"fare (skew {fare.skew():.1f})")
sns.histplot(log_fare, bins=40, ax=axes[1]); axes[1].set_title(f"log1p(fare) (skew {log_fare.skew():.1f})")
plt.tight_layout(); plt.show()

---
# Part 4 — LIVE DEMO 1: normalisation vs standardisation on real features

Many models compare features by **distance** (k-NN, k-means, SVM) or learn with **gradients** (logistic regression, neural networks). If one feature is measured in thousands and another in decimals, the big one dominates purely because of its units.

| Method | Formula | Result | Use when |
|---|---|---|---|
| **Standardisation** (z-score) | (x − mean) / std | mean 0, std 1, no fixed range | the default for most models |
| **Normalisation** (min–max) | (x − min) / (max − min) | exactly 0 to 1 | you need a bounded range (pixels, some neural nets) |
| **Robust scaling** | (x − median) / IQR | centred on the median | features with strong outliers |

Trees (decision trees, Random Forest, XGBoost) split on one feature at a time and **don't need scaling**.

**The dataset:** 569 breast-tumour biopsies (built into scikit-learn), 30 measurements each, label = malignant or benign. Look at the scales.

In [ ]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target            # target: 1 = benign, 0 = malignant
print(X.shape)
X[["mean area", "mean radius", "mean smoothness", "mean fractal dimension"]].describe().loc[["mean", "std", "min", "max"]].round(3)

`mean area` is in the hundreds to thousands; `mean smoothness` is around 0.1. To a distance-based model, a difference of 100 in area looks **thousands of times** more important than any difference in smoothness, which is purely an accident of units.

**The right procedure** (this is where many projects leak):
1. Split into train and test **first**.
2. `fit` the scaler on the **training** data only (it learns the means and stds).
3. `transform` both train and test with those training statistics.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_train)            # learns 30 means and 30 stds from TRAIN only
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)                # test uses the TRAIN statistics
print("train after scaling: mean", X_train_s.mean().round(3), "std", X_train_s.std().round(3))
print("test after scaling:  mean", X_test_s.mean().round(3), "std", X_test_s.std().round(3), "<- close to, not exactly, 0 and 1: correct!")

In [ ]:
results = {}
for name, scaler in [("no scaling", None), ("standardise", StandardScaler()),
                     ("min-max normalise", MinMaxScaler()), ("robust", RobustScaler())]:
    steps = [scaler, KNeighborsClassifier(n_neighbors=5)] if scaler else [KNeighborsClassifier(n_neighbors=5)]
    model = make_pipeline(*steps).fit(X_train, y_train)   # the pipeline fits the scaler on train only
    results[name] = model.score(X_test, y_test)
pd.Series(results, name="k-NN test accuracy").round(3)

Same model, same data, same neighbours: scaling alone lifts k-NN accuracy by several points. Putting the scaler inside a `Pipeline` guarantees it's only ever fitted on training data, including during cross-validation (Day 9).

Now do it by hand once, so the formulas aren't magic:

In [ ]:
col = X_train["mean area"]
z_by_hand = (col - col.mean()) / col.std(ddof=0)            # StandardScaler uses ddof=0
mm_by_hand = (col - col.min()) / (col.max() - col.min())
print("matches StandardScaler:", np.allclose(z_by_hand, StandardScaler().fit_transform(X_train[["mean area"]]).ravel()))
print("matches MinMaxScaler:  ", np.allclose(mm_by_hand, MinMaxScaler().fit_transform(X_train[["mean area"]]).ravel()))
print("scaling doesn't change the SHAPE: skew before", round(col.skew(), 3), "after", round(pd.Series(z_by_hand).skew(), 3))

Note the last line: **scaling changes units, not shape.** A skewed feature stays exactly as skewed after standardising. To change the shape, use a transform like `log1p` (Part 3).

---
# Part 5 — Population vs sample

- **Population**: everyone you care about (all Indian voters, all future emails).
- **Sample**: the part you actually measured.
- **Parameter** (μ, σ, p): a true value of the population, usually unknown.
- **Statistic** (x̄, s, p̂): computed from the sample, used to *estimate* the parameter.

Statistics is about how far a statistic can be trusted as an estimate of the parameter.

### Sampling methods
| Method | How | ML example |
|---|---|---|
| Simple random | every member equally likely | `df.sample(frac=0.2)` |
| **Stratified** | random within each group, keeping group proportions | `train_test_split(..., stratify=y)` |
| Systematic | every k-th item | every 10th log line |
| Cluster | pick whole groups at random | survey 20 random schools, everyone in them |

### Sampling bias: a big sample doesn't fix a biased one
- **Selection bias**: training a loan model only on applicants who were approved.
- **Survivorship bias**: studying only companies that still exist.
- **Non-response bias**: only angry customers answer the survey.

In [ ]:
# Stratified vs simple random split on an imbalanced label: watch the minority share wobble
y_rare = (rng.random(1000) < 0.05).astype(int)          # 5% positives
plain, strat = [], []
for seed in range(200):
    _, te = train_test_split(y_rare, test_size=0.2, random_state=seed)
    _, te_s = train_test_split(y_rare, test_size=0.2, random_state=seed, stratify=y_rare)
    plain.append(te.mean()); strat.append(te_s.mean())
print(f"positive share in test set, random split    : {min(plain):.3f} to {max(plain):.3f}")
print(f"positive share in test set, stratified split: {min(strat):.3f} to {max(strat):.3f}")

---
# Part 6 — LIVE DEMO 2: the Central Limit Theorem by simulation

**The claim:** take many samples of size n from *any* population with mean μ and std σ, and compute each sample's mean. Those **sample means** are approximately **normally distributed**, centred on μ, with standard deviation

$$\text{SE} = \frac{\sigma}{\sqrt{n}} \quad \text{(the standard error)}$$

even if the population itself is wildly non-normal. This one theorem is why confidence intervals and t-tests work on real data.

**Our population:** the 891 Titanic fares, which are about as non-normal as data gets (skew 4.8).

In [ ]:
population = titanic["fare"].to_numpy()
mu, sigma = population.mean(), population.std()          # population: divide by N
print(f"population mean {mu:.2f}, std {sigma:.2f}, skew {stats.skew(population):.2f}")

fig, axes = plt.subplots(1, 4, figsize=(17, 3.4))
for ax, n in zip(axes, [1, 5, 30, 100]):
    means = rng.choice(population, size=(10_000, n), replace=True).mean(axis=1)
    sns.histplot(means, bins=50, stat="density", ax=ax)
    grid = np.linspace(means.min(), means.max(), 200)
    ax.plot(grid, stats.norm.pdf(grid, mu, sigma / np.sqrt(n)), color="red", label="CLT prediction")
    ax.set_title(f"n = {n}: skew {stats.skew(means):.2f}")
axes[0].legend()
plt.suptitle("Means of 10,000 samples of Titanic fares", y=1.04); plt.show()

In [ ]:
print(f"{'n':>4} | {'std of sample means':>20} | {'sigma / sqrt(n)':>15}")
for n in [5, 30, 100, 400]:
    means = rng.choice(population, size=(20_000, n), replace=True).mean(axis=1)
    print(f"{n:>4} | {means.std():>20.2f} | {sigma / np.sqrt(n):>15.2f}")

What the simulation shows:
1. **n = 1** is just the population: very skewed.
2. As n grows, the histogram of means becomes a **bell**, and the red CLT curve fits it. For very skewed data like this you need n ≈ 30–100; for mildly skewed data, n = 30 is plenty.
3. The spread shrinks exactly like **σ/√n**. To halve the uncertainty you need **4×** the data, not 2×.

**Why ML engineers care:** a model's test accuracy is an average of 0/1 "correct" values, so by the CLT it has a standard error of about √(p(1−p)/n). With 7,133 test messages and 97.95% accuracy (TextPulse), SE ≈ 0.17%. With 200 test rows it would be about 1%, so a 1-point "improvement" could be pure noise.

In [ ]:
for n_test in [200, 1000, 7133]:
    p = 0.9795
    print(f"accuracy 97.95% on {n_test:>5} test rows -> standard error {100*np.sqrt(p*(1-p)/n_test):.2f} points")

---
# Part 7 — Confidence intervals

A point estimate ("average tip = 3.00") says nothing about how sure we are. A **confidence interval** gives a range:

$$\bar{x} \pm t^{*} \cdot \frac{s}{\sqrt{n}}$$

- x̄: sample mean; s: sample std; n: sample size; s/√n: standard error.
- t*: the multiplier for the confidence level, from the **t-distribution** with n − 1 degrees of freedom (≈ 1.96 for 95% when n is large). We use t instead of the normal's 1.96 because s is itself estimated, which adds uncertainty for small samples.

In [ ]:
tip = tips["tip"]
n, xbar, s = len(tip), tip.mean(), tip.std(ddof=1)
se = s / np.sqrt(n)
t_star = stats.t.ppf(0.975, df=n - 1)
print(f"n = {n}, mean tip = {xbar:.3f}, s = {s:.3f}, SE = {se:.3f}, t* = {t_star:.3f}")
print(f"95% CI by hand : ({xbar - t_star*se:.3f}, {xbar + t_star*se:.3f})")
print("95% CI by scipy:", tuple(round(v, 3) for v in stats.t.interval(0.95, df=n-1, loc=xbar, scale=se)))

### What "95% confident" really means
**Wrong:** "there's a 95% probability the true mean is in *this* interval." (The true mean is fixed; it's either in it or not.)
**Right:** "if we repeated the sampling many times, **95% of the intervals built this way would contain the true mean**." The 95% describes the *method*.

Prove it: sample from a population where we know the truth, build 1,000 intervals and count how many catch μ.

In [ ]:
true_mu, n = 50, 25
hits = 0
fig, ax = plt.subplots(figsize=(12, 3.5))
for i in range(1000):
    sample = rng.exponential(scale=true_mu, size=n)            # skewed population, true mean 50
    lo, hi = stats.t.interval(0.95, df=n-1, loc=sample.mean(), scale=sample.std(ddof=1)/np.sqrt(n))
    caught = lo <= true_mu <= hi
    hits += caught
    if i < 100:
        ax.plot([i, i], [lo, hi], color="steelblue" if caught else "red")
ax.axhline(true_mu, color="black", linestyle="--"); ax.set_title("First 100 of 1,000 intervals (red = missed the true mean)")
plt.show()
print(f"{hits / 10:.1f}% of 1,000 intervals contained the true mean")

About 93%: close to, but a little below, the promised 95%. Look at the chart: the red misses are almost all **below** the true mean, because a small sample from skewed data usually misses the rare huge values that pull the true mean up. The t-interval assumes the sample mean is normal; with a skewed (exponential) population and only n = 25, the CLT hasn't fully kicked in yet. Coverage improves only slowly with n for skewed data (about 94% at n = 100), which is the practical meaning of "the CLT is an approximation". For skewed data, a bootstrap interval (below) is often more trustworthy.

### Bootstrap: a confidence interval for anything
No formula for the CI of a median, or of a model's F1 score? **Resample your data with replacement** thousands of times, compute the statistic each time, and take the middle 95% of the results.

In [ ]:
boot_medians = [np.median(rng.choice(tip, size=len(tip), replace=True)) for _ in range(10_000)]
lo, hi = np.percentile(boot_medians, [2.5, 97.5])
print(f"median tip = {tip.median():.2f}, 95% bootstrap CI ({lo:.2f}, {hi:.2f})")
print("scipy does it too:", stats.bootstrap((tip.to_numpy(),), np.median, random_state=4).confidence_interval)

---
# Part 8 — Hypothesis tests

A hypothesis test asks: **could this pattern just be luck?**

1. **Null hypothesis H₀**: no effect, no difference ("smokers and non-smokers tip the same").
2. **Alternative H₁**: there is a difference.
3. Pick a significance level **α** (usually 0.05) *before* looking.
4. Compute a **test statistic** from the data and its **p-value**: *the probability of seeing a result at least this extreme **if H₀ were true**.*
5. If p < α, **reject H₀** ("statistically significant"). Otherwise, **fail to reject**. That's not the same as proving H₀ true.

### The two ways to be wrong
| | H₀ really true | H₀ really false |
|---|---|---|
| **Reject H₀** | **Type I error** (false positive), probability α | correct: **power** = 1 − β |
| **Don't reject** | correct | **Type II error** (false negative), probability β |

The same trade-off as precision vs recall: lowering α means fewer false alarms but more missed effects. **Power** goes up with bigger samples and bigger effects.

### The p-value, built from scratch: a permutation test
Do dinner bills have larger tips than lunch bills? If time of day didn't matter (H₀), the "dinner"/"lunch" labels would be interchangeable. So shuffle the labels thousands of times and see how often chance alone produces a gap as big as the real one.

In [ ]:
dinner = tips.loc[tips.time == "Dinner", "tip"].to_numpy()
lunch = tips.loc[tips.time == "Lunch", "tip"].to_numpy()
observed = dinner.mean() - lunch.mean()
pooled = np.concatenate([dinner, lunch])

gaps = []
for _ in range(10_000):
    rng.shuffle(pooled)                                        # H0: labels don't matter
    gaps.append(pooled[:len(dinner)].mean() - pooled[len(dinner):].mean())
gaps = np.array(gaps)
p_perm = (np.abs(gaps) >= abs(observed)).mean()                # two-sided
print(f"observed gap: {observed:.3f} | permutation p-value: {p_perm:.4f}")
print("Welch t-test p-value:", round(stats.ttest_ind(dinner, lunch, equal_var=False).pvalue, 4))

plt.figure(figsize=(8, 3)); sns.histplot(gaps, bins=60)
plt.axvline(observed, color="red", label="observed"); plt.axvline(-observed, color="red", linestyle="--")
plt.title("Gaps produced by chance alone (H0)"); plt.legend(); plt.show()

The p-value is the share of the chance-only histogram that lies beyond the red lines. Here the permutation test gives **p ≈ 0.06** and the Welch t-test **p ≈ 0.04**: right on the 0.05 line, and the two methods even land on opposite sides of it.

The honest conclusion is **weak evidence** that dinner tips are higher, not a discovery. Dinner bills are also bigger, so a higher tip in rupees may just follow the bill (Exercise 4 looks at tip *percentage*). A borderline p-value is a reason to collect more data, never a reason to tweak the analysis until it crosses 0.05.

### Choosing the right test
| Question | Data | Test | scipy |
|---|---|---|---|
| Is the mean different from a value? | one numeric sample | one-sample t | `stats.ttest_1samp` |
| Do two groups differ? | numeric, 2 independent groups | **Welch t-test** | `stats.ttest_ind(a, b, equal_var=False)` |
| Did the same people change? | numeric, before/after pairs | paired t | `stats.ttest_rel` |
| Do 3+ groups differ? | numeric, 3+ groups | one-way ANOVA | `stats.f_oneway` |
| Are two categories related? | counts in a table | chi-square | `stats.chi2_contingency` |
| Do two conversion rates differ? | two proportions | two-proportion z-test | by hand (Part 9) |
| Data very skewed, small n? | numeric | Mann–Whitney U (rank-based) | `stats.mannwhitneyu` |

In [ ]:
# One-sample: is the average tip different from 15% of the bill?
tip_pct = tips["tip"] / tips["total_bill"]
print("mean tip % =", round(tip_pct.mean() * 100, 2), "| one-sample t vs 15%:", stats.ttest_1samp(tip_pct, 0.15))

# ANOVA: does the tip % differ by day?
groups = [g["tip"] / g["total_bill"] for _, g in tips.groupby("day", observed=True)]
print("ANOVA by day:", stats.f_oneway(*groups))

# Chi-square: are smokers more common at dinner than lunch?
table = pd.crosstab(tips["smoker"], tips["time"])
chi2, p, dof, expected = stats.chi2_contingency(table)
print(table, f"\nchi2 = {chi2:.2f}, p = {p:.3f}")

In [ ]:
# Paired t-test: the same 20 students before and after a revision course (simulated)
before = rng.normal(62, 10, 20).round()
after = (before + rng.normal(4, 5, 20)).round()               # true average gain: 4 marks
print("paired t-test  :", stats.ttest_rel(after, before))
print("unpaired t-test:", stats.ttest_ind(after, before))       # wrong test for paired data: much weaker

The paired test is far more sensitive because it compares each student with **themselves**, removing the big differences *between* students. Using the wrong test throws that information away.

### Significant ≠ important: effect size
With huge samples, tiny differences become "significant". Always report **how big** the effect is. **Cohen's d** = difference in means ÷ pooled std: 0.2 small, 0.5 medium, 0.8 large.

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_sd = np.sqrt(((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2))
    return (np.mean(a) - np.mean(b)) / pooled_sd

a = rng.normal(100.0, 15, 200_000)
b = rng.normal(100.3, 15, 200_000)           # a real but tiny difference
print("p-value  :", stats.ttest_ind(a, b).pvalue)
print("Cohen's d:", round(cohens_d(b, a), 3), "<- negligible in practice")

### The multiple-testing trap
Test 20 useless features at α = 0.05 and you expect about **one** to come out "significant" by luck. Fixes: the **Bonferroni** correction (use α / number of tests), or treat exploratory findings as hypotheses to confirm on new data.

In [ ]:
false_alarms = 0
for _ in range(1000):                                        # 1000 studies, each testing 20 useless features
    pvals = [stats.ttest_ind(rng.normal(0, 1, 30), rng.normal(0, 1, 30)).pvalue for _ in range(20)]
    false_alarms += min(pvals) < 0.05
print(f"studies with at least one 'significant' useless feature: {false_alarms / 10:.1f}%  (theory: 1 - 0.95^20 = {100*(1-0.95**20):.1f}%)")

---
# Part 9 — An A/B test, end to end

Your team changes the "Sign up" button. **A** (old) and **B** (new) are each shown to a random half of visitors.

**Before the test**, decide: the metric (conversion rate), α = 0.05, and the sample size needed to detect the smallest lift you care about. Stopping the moment the p-value dips below 0.05 ("peeking") inflates false positives.

Two-proportion z-test:
$$z = \frac{\hat p_B - \hat p_A}{\sqrt{\hat p(1-\hat p)\left(\frac{1}{n_A} + \frac{1}{n_B}\right)}}, \qquad \hat p = \frac{x_A + x_B}{n_A + n_B}$$

In [ ]:
n_A, x_A = 10_000, 420      # visitors, sign-ups
n_B, x_B = 10_000, 485

p_A, p_B = x_A / n_A, x_B / n_B
p_pool = (x_A + x_B) / (n_A + n_B)
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
z = (p_B - p_A) / se_pool
p_value = 2 * stats.norm.sf(abs(z))

se_diff = np.sqrt(p_A*(1-p_A)/n_A + p_B*(1-p_B)/n_B)        # unpooled SE for the interval
ci = (p_B - p_A - 1.96*se_diff, p_B - p_A + 1.96*se_diff)

print(f"conversion A {p_A:.2%}  B {p_B:.2%}  lift {p_B - p_A:+.2%} ({(p_B/p_A - 1):+.1%} relative)")
print(f"z = {z:.2f}, p = {p_value:.4f}")
print(f"95% CI for the lift: {ci[0]:+.2%} to {ci[1]:+.2%}")

In [ ]:
# How many visitors per group would we need? (80% power, alpha 0.05, to detect 4.2% -> 4.7%)
from math import ceil
p1, p2 = 0.042, 0.047
z_a, z_b = stats.norm.ppf(0.975), stats.norm.ppf(0.80)
p_bar = (p1 + p2) / 2
n_needed = ((z_a * np.sqrt(2*p_bar*(1-p_bar)) + z_b * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2) / (p2 - p1)**2
print("visitors needed per group:", ceil(n_needed))

**How to report it to a manager:** "The new button raised sign-ups from 4.20% to 4.85% (+15% relative). The difference is statistically significant (p ≈ 0.03), and we're 95% confident the true lift is between about +0.07 and +1.23 points."

Note the honest caveat hidden in that interval: the lift could be as small as 0.07 points. That's also why the sample-size calculation matters: 10,000 per group is a bit **less** than the ~27,000 needed for 80% power to detect a 0.5-point lift, so this test was somewhat lucky to reach significance.

---
# Part 10 — Exercises
Paste your code in chat for review.

**Ex 1 — Describe.** For `tips["total_bill"]`: compute the mean, median, std (both ddof=0 and 1), IQR, skewness, and the number of IQR outliers. In two sentences, say which centre and spread you'd report and why.

**Ex 2 — Scaling.** On the breast-cancer data, compare k-NN (k = 5) test accuracy with **no scaling**, `StandardScaler` and `MinMaxScaler`, but this time scale **wrongly** (fit the scaler on all of X before splitting). Does the leak change the accuracy here? Explain why it's still wrong.

**Ex 3 — CLT.** Use a population of 100,000 values from `rng.exponential(scale=10)`. For n = 2, 10, 50, draw 5,000 sample means and print their mean, std and skew. Compare the std with σ/√n.

**Ex 4 — Test.** Do smokers and non-smokers give a different tip **percentage**? State H₀ and H₁, run a Welch t-test and a permutation test, report Cohen's d, and write a one-line conclusion for a manager.

**Stretch.** Re-run the A/B test from Part 9 as a **simulation**: assume both buttons truly convert at 4.5%, simulate 2,000 A/B tests of 10,000 visitors per group, and count how often p < 0.05. What should that rate be, and why?

In [ ]:
# Ex 1

In [ ]:
# Ex 2

In [ ]:
# Ex 3

In [ ]:
# Ex 4

In [ ]:
# Stretch

---
### Day 4 recap
- **Centre:** mean (uses everything, pulled by outliers), median (robust; use for skewed data), mode (categories).
- **Spread:** std is in original units; sample variance divides by **n − 1**; IQR is the robust choice. Outliers: IQR rule for skewed data, |z| > 3 for normal-ish data.
- **Shape:** right skew → `log1p`. Scaling changes units, **not** shape.
- **Standardise** (z-score) by default, **min–max** for a fixed 0–1 range, **robust** for outliers; trees need none. **Fit scalers on training data only**, ideally inside a Pipeline.
- **Sampling:** stratify imbalanced labels; a biased sample stays biased however large it is.
- **CLT:** sample means are ≈ normal with SE = σ/√n; 4× the data halves the uncertainty.
- **95% CI** = x̄ ± t*·s/√n; "95%" describes the method's long-run hit rate. **Bootstrap** gives a CI for any statistic.
- **p-value** = P(data this extreme | H₀ true). Small p → reject H₀. Report the **effect size** too, and beware peeking and multiple tests.
- **Tomorrow (Day 5): Linear algebra.** Vectors, norms, dot products, matrices, eigenvectors: the language models compute in.